# Aria Troubleshooting Decision Tree & Quick Fixes

**Fast diagnostic flowcharts and step-by-step fixes for common issues.**

When something breaks, follow the relevant decision tree to find the root cause quickly.


## Issue Category: "Chat Not Working"

```
START: Chat endpoint returns error or wrong response
    ↓
Is the error 500 (server error)?
    ├─→ YES: Continue to "Server Error" path
    └─→ NO: Continue to "Client Error" path

SERVER ERROR PATH (500):
    ├─ Check logs: tail -f data_out/function_app.log
    ├─ Error says "provider not found"?
    │   ├─→ YES: → Go to "Provider Detection Issue"
    │   └─→ NO: Continue
    ├─ Error says "database error"?
    │   ├─→ YES: → Go to "Database Connection Issue"
    │   └─→ NO: Continue
    ├─ Error says "memory error"?
    │   ├─→ YES: → Go to "Memory Issue"
    │   └─→ NO: Continue
    └─ Other error?
        └─→ Search docs/REPOSITORY_ARCHITECTURE.md Part 5 for error message

CLIENT ERROR PATH (4xx):
    ├─ Check request format
    ├─ Verify JSON is valid: echo '{"message":"test"}' | python -m json.tool
    ├─ Check Content-Type: should be "application/json"
    ├─ Verify required fields present (message, session_id if needed)
    └─ Try with curl to isolate issue from client
```

---

## Issue: Provider Detection Not Working

### Symptoms

- Chat always returns "Local provider (echo mode)"
- Can't connect to Azure OpenAI
- LM Studio / Ollama not being detected

### Quick Diagnostic

```bash
# Step 1: Check what's actually running
netstat -tuln | grep -E "1234|11434"  # Check LM Studio / Ollama
echo $LMSTUDIO_BASE_URL               # Check env var
echo $AZURE_OPENAI_API_KEY            # Check Azure key (should show *** if set)

# Step 2: Check detection function
curl http://localhost:7071/api/ai/status | jq .provider

# Step 3: Test each provider manually
# LM Studio
curl http://127.0.0.1:1234/v1/models 2>/dev/null | head

# Ollama
curl http://127.0.0.1:11434/api/tags 2>/dev/null | head

# Azure OpenAI
curl -X POST "https://{resource}.openai.azure.com/openai/deployments/{deployment}/chat/completions?api-version=2024-02-15-preview" \
  -H "api-key: $AZURE_OPENAI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{"messages":[{"role":"user","content":"test"}]}' 2>/dev/null | head
```

### Fix Checklist

**LM Studio / Ollama not detected:**

- [ ] Is the service actually running? (Check with netstat/lsof)
- [ ] Is it on the right port? (LM Studio: 1234, Ollama: 11434)
- [ ] Is the URL correct? Check `LMSTUDIO_BASE_URL` or `OLLAMA_BASE_URL` env vars
- [ ] Try restarting the service
- [ ] Check for firewall blocking localhost

**Azure OpenAI not detected:**

- [ ] Is `AZURE_OPENAI_API_KEY` set? (Check: `echo $AZURE_OPENAI_API_KEY | wc -c`, should be >50)
- [ ] Is `AZURE_OPENAI_ENDPOINT` set correctly?
- [ ] Is `AZURE_OPENAI_DEPLOYMENT` set correctly?
- [ ] Is `AZURE_OPENAI_API_VERSION` set? (default: 2024-02-15-preview)
- [ ] Test: `curl -X POST $AZURE_OPENAI_ENDPOINT/...` manually
- [ ] Check: Are credentials expired? Refresh in Azure portal

**All providers failing (falls back to local echo):**

- [ ] No services running locally
- [ ] No Azure credentials set
- [ ] Network issues (can't reach cloud)
- [ ] All environments have been cleared

### Recovery Steps

```bash
# 1. Reset environment
unset LMSTUDIO_BASE_URL OLLAMA_BASE_URL
export LMSTUDIO_BASE_URL="http://127.0.0.1:1234/v1"
export OLLAMA_BASE_URL="http://127.0.0.1:11434/v1"

# 2. Start a provider locally
# Option A: LM Studio (GUI)
# Download from lmstudio.ai, open app, load a model, keep running

# Option B: Ollama
ollama serve
# In another terminal:
ollama pull llama2

# 3. Test detection
curl http://localhost:7071/api/ai/status | jq .provider
# Should now show provider name instead of "local"

# 4. Test chat
curl -X POST http://localhost:7071/api/chat \
  -H "Content-Type: application/json" \
  -d '{"message":"test"}'
```

---

## Issue: Database Connection Failing

### Symptoms

- Chat works for 1-2 messages then stops
- Error: "database pool exhausted"
- Error: "connection timeout"
- Status shows "database: unhealthy"

### Quick Diagnostic

```bash
# Step 1: Check database status
curl http://localhost:7071/api/ai/status | jq .database

# Step 2: Check what database is configured
echo $QAI_DB_CONN      # Should show connection string
echo $QAI_SQL_POOL_SIZE  # Should show pool size (default 10)

# Step 3: Verify database file exists (if using SQLite)
ls -la data_out/*.db   # Should show database file

# Step 4: Test database directly
# For SQLite
sqlite3 $(echo $QAI_DB_CONN | sed 's|sqlite:///||') "SELECT COUNT(*) FROM sqlite_master;"

# For PostgreSQL
psql $QAI_DB_CONN -c "SELECT 1;"
```

### Fix Checklist

**Database pool saturated:**

- [ ] Current pool size too small
- [ ] Many connections waiting for responses
- [ ] Slow queries blocking the pool

**Fix:**

```bash
# Increase pool size
export QAI_SQL_POOL_SIZE=20  # Increase from default 10
# Restart: func host start --port 7071
```

**Database file locked (SQLite):**

- [ ] Another process has the file open
- [ ] Corrupted database
- [ ] Wrong permissions

**Fix:**

```bash
# Check what's using the file
lsof data_out/*.db

# Kill other processes if needed
# Or backup and delete corrupted database
mv data_out/app.db data_out/app.db.backup
# New database will be created on next run
```

**Connection timeout (PostgreSQL/Azure SQL):**

- [ ] Network unreachable
- [ ] Database server down
- [ ] Wrong credentials
- [ ] Firewall blocking

**Fix:**

```bash
# Test connectivity
telnet $(echo $QAI_DB_CONN | grep -oP 'host=\K[^;]*') \
       $(echo $QAI_DB_CONN | grep -oP 'port=\K[^;]*')

# Or use psql
psql $QAI_DB_CONN -c "SELECT 1;" -v ON_ERROR_STOP=1
```

### Recovery Steps

```bash
# 1. Check current status
curl http://localhost:7071/api/ai/status | jq .

# 2. If using SQLite
# - Backup old database
# - Delete corrupted database
mv data_out/app.db data_out/app.db.backup

# 3. If using PostgreSQL/Azure SQL
# - Verify connection string in $QAI_DB_CONN
# - Test connectivity manually
# - Ensure database exists

# 4. Restart Functions
func host start --port 7071

# 5. Verify database healthy
curl http://localhost:7071/api/ai/status | jq .database.available
# Should be true
```

---

## Issue: Aria Character Not Responding to Commands

### Symptoms

- Aria doesn't move / respond to commands
- Command parsing always returns empty actions
- API returns error 500 or 400

### Quick Diagnostic

```bash
# Step 1: Check Aria server is running
curl http://localhost:8080/api/aria/health | jq

# Step 2: Check current state
curl http://localhost:8080/api/aria/state | jq

# Step 3: Send a simple command
curl -X POST http://localhost:8080/api/aria/command \
  -H "Content-Type: application/json" \
  -d '{"command":"wave"}'

# Step 4: Check action schema (what commands are valid?)
curl http://localhost:8080/api/aria/schema | jq .actions
```

### Fix Checklist

**Aria server not running:**

- [ ] Is server process running? `ps aux | grep "python.*server.py"`
- [ ] Is it on port 8080? `lsof -i :8080`
- [ ] Check logs: `tail -50 app_output.log` (if running with output redirection)

**Command not recognized:**

- [ ] Check valid gestures: `curl http://localhost:8080/api/aria/schema | jq .gestures`
- [ ] Check valid positions: `curl http://localhost:8080/api/aria/schema | jq .positions`
- [ ] Try simple commands first: "wave", "move left", "jump"
- [ ] Check JSON syntax is valid

**LLM provider not available:**

- [ ] Check if chat provider is available
- [ ] Falls back to rule-based parser (legacy tags like `[aria:gesture:wave]`)
- [ ] Check: `curl http://localhost:7071/api/ai/status | jq .provider`

### Recovery Steps

```bash
# 1. Start Aria server
cd apps/aria
python server.py
# Should output: "Running on http://0.0.0.0:8080"

# 2. Verify server responding
curl http://localhost:8080/api/aria/health

# 3. Test simple command
curl -X POST http://localhost:8080/api/aria/command \
  -H "Content-Type: application/json" \
  -d '{"command":"wave"}'

# 4. If still failing, check logs
# Server outputs directly to terminal (unless redirected)
```

---

## Issue: Quantum Jobs Timing Out or Costing Too Much

### Symptoms

- Quantum requests timeout (>1 minute)
- Real QPU jobs submitted when only simulator intended
- Cost estimation shows millions of dollars

### Quick Diagnostic

```bash
# Step 1: Check quantum configuration
cat config/quantum_autorun.yaml | grep -E "simulator|cost|timeout"

# Step 2: Check environment variables
echo $QUANTUM_PROVIDER         # Should be "simulator"
echo $AZURE_CONFIRM_COST       # Should be "false"
echo $QUANTUM_MAX_SHOTS        # Should be reasonable (100-1000)

# Step 3: Check MCP server status (if using quantum via MCP)
curl http://localhost:9001/health 2>/dev/null | jq

# Step 4: Dry-run quantum orchestrator
python scripts/quantum_autorun.py --dry-run
```

### Fix Checklist

**Jobs using real QPU when simulator intended:**

- [ ] Set `QUANTUM_PROVIDER=simulator` in env
- [ ] Set `AZURE_CONFIRM_COST=false` in env
- [ ] Update `config/quantum_autorun.yaml`: `simulator_only: true`
- [ ] Restart MCP server: `python ai-projects/quantum-ml/quantum_mcp_server.py`

**Jobs timing out:**

- [ ] Circuit too complex (depth > 50 qubits)
- [ ] Too many shots (> 10,000)
- [ ] Using real QPU instead of simulator
- [ ] Network connectivity issue

**Fix:**

```yaml
# config/quantum_autorun.yaml
simulator_only: true # Force simulator
max_shots: 100 # Reduce shots
circuit_depth_limit: 20 # Keep circuits simple
azure_confirm_cost: false # Disable real QPU
```

### Recovery Steps

```bash
# 1. Set safety constraints
export QUANTUM_PROVIDER="simulator"
export AZURE_CONFIRM_COST="false"
export QUANTUM_MAX_SHOTS="100"

# 2. Update config file
cat > config/quantum_dev.yaml << 'EOF'
simulator_only: true
max_shots: 100
circuit_depth_limit: 20
azure_confirm_cost: false
EOF

# 3. Dry-run to validate
python scripts/quantum_autorun.py --dry-run
# Should show: "All jobs using simulator (no cost)"

# 4. Test simple quantum circuit
python -c "
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)

sim = AerSimulator()
job = sim.run(circuit, shots=100)
print(job.result().get_counts())
"
```

---

## Issue: Training Stuck or Not Starting

### Symptoms

- Training cycles not starting
- No new models being saved
- Status file not updating
- Log file shows errors

### Quick Diagnostic

```bash
# Step 1: Check if process is running
ps aux | grep autonomous_training_orchestrator

# Step 2: Check logs
tail -50 data_out/autonomous_training.log

# Step 3: Check status file
cat data_out/autonomous_training_status.json | jq

# Step 4: Check datasets exist
ls -la datasets/quantum datasets/chat datasets/massive_quantum
```

### Fix Checklist

**Process not running:**

- [ ] Start process: `python scripts/autonomous_training_orchestrator.py`
- [ ] Check output for errors
- [ ] Verify datasets exist: `ls datasets/*/`

**Datasets missing:**

- [ ] Create datasets folder: `mkdir -p datasets/quantum datasets/chat`
- [ ] Add sample data (must be JSONL format for training)
- [ ] Validate dataset format: `python scripts/validate_datasets.py`

**Process running but stuck:**

- [ ] Check if waiting for 30min cycle: `cat data_out/autonomous_training_status.json | jq .last_cycle_start`
- [ ] Force immediate cycle: `pkill -USR1 -f autonomous_training_orchestrator.py`
- [ ] Check if stalled: kill and restart process

### Recovery Steps

```bash
# 1. Check status
cat data_out/autonomous_training_status.json | jq

# 2. If stuck, kill and restart
pkill -TERM -f autonomous_training_orchestrator.py
sleep 2

# 3. Verify datasets
python scripts/validate_datasets.py

# 4. Start fresh
python scripts/autonomous_training_orchestrator.py

# 5. Monitor
tail -f data_out/autonomous_training.log
```

---

## Quick Reference: System Health Commands

```bash
# Full system health check
curl http://localhost:7071/api/ai/status | jq

# Provider detection
curl http://localhost:7071/api/ai/status | jq .provider

# Database status
curl http://localhost:7071/api/ai/status | jq .database

# Quantum backend
curl http://localhost:7071/api/quantum-llm/status | jq

# Check if Aria server is running
curl http://localhost:8080/api/aria/health | jq

# List running Python processes
ps aux | grep -E "function_app|aria|quantum|train" | grep -v grep

# Check network connectivity
netstat -tuln | grep -E "1234|11434|7071|8080"

# View recent logs
tail -f data_out/autonomous_training.log
tail -f logs/*.log 2>/dev/null || echo "No logs yet"

# Monitor resource usage
python scripts/resource_monitor.py --snapshot
```


## When to Escalate Issues

**Contact team if:**

- Error is not in this troubleshooting guide
- Same fix doesn't work after 2 attempts
- Issue affects multiple team members
- Possible data loss (database corruption)
- Security issue (credentials exposed)

**Before escalating, provide:**

- Error message (full stack trace)
- Steps to reproduce
- System info: `uname -a`, `python --version`
- Relevant logs: `data_out/*.log`
- What you've already tried

**Documentation to reference:**

- Specific error: Search REPOSITORY_ARCHITECTURE.md Part 5
- General issue: Check KEYWORD_INDEX.md
- Process questions: See ARCHITECTURE_DIAGRAMS.md
- Code examples: ARIA_REPOSITORY_INTERACTIVE_GUIDE.ipynb


## When to Escalate Issues

**Contact team if:**

- Error is not in this troubleshooting guide
- Same fix doesn't work after 2 attempts
- Issue affects multiple team members
- Possible data loss (database corruption)
- Security issue (credentials exposed)

**Before escalating, provide:**

- Error message (full stack trace)
- Steps to reproduce
- System info: `uname -a`, `python --version`
- Relevant logs: `data_out/*.log`
- What you've already tried

**Documentation to reference:**

- Specific error: Search REPOSITORY_ARCHITECTURE.md Part 5
- General issue: Check KEYWORD_INDEX.md
- Process questions: See ARCHITECTURE_DIAGRAMS.md
- Code examples: ARIA_REPOSITORY_INTERACTIVE_GUIDE.ipynb
